In [ ]:
import re
from pathlib import Path
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt


In [2]:
FEATURES_PATH = Path(
    "/Users/ilg/Desktop/year4/M4R/python_files/eda_output_unnormalised/03_sigmoid_normalisation/features_sigmoid_normalised.csv"
)

OUT_PATH = Path(
    "/Users/ilg/Desktop/year4/M4R/python_files/some_other_eda/loading_family_analysis/feature_family_mapping.csv"
)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)


def parse_pyspoc_feature(feature: str) -> dict:
    f = str(feature)

    result = {
        "feature": f,
        "pipeline_type": "unknown",
        "statistic_module": "",
        "statistic_family": "",
        "statistic_output": "",
        "reducer_module": "",
        "reducer_family": "",
        "reducer_output": "",
    }

    # Algorithm-specific metrics outside PySPoC
    low = f.lower()
    if "fabia" in low or "hoyer" in low or "snr" in low:
        result.update({
            "pipeline_type": "algorithm_metric",
            "statistic_family": "FABIA metrics",
            "reducer_family": "self",
        })
        return result

    if "plaid" in low or "wtve" in low or "bic" in low:
        result.update({
            "pipeline_type": "algorithm_metric",
            "statistic_family": "Plaid metrics",
            "reducer_family": "self",
        })
        return result

    # Statistic + reducer:
    # pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Diag.matrix_diagonal_entries_2
    if "pyspoc.statistics." in f and ".pyspoc.reducers." in f:
        stat_part, red_part = f.split(".pyspoc.reducers.", 1)
        stat_part = stat_part.replace("pyspoc.statistics.", "")

        stat_tokens = stat_part.split(".")
        red_tokens = red_part.split(".")

        result["pipeline_type"] = "statistic_reducer"

        if len(stat_tokens) >= 2:
            result["statistic_module"] = stat_tokens[0]
            result["statistic_family"] = stat_tokens[1]
            result["statistic_output"] = ".".join(stat_tokens[2:])

        if len(red_tokens) >= 2:
            result["reducer_module"] = red_tokens[0]
            result["reducer_family"] = red_tokens[1]
            result["reducer_output"] = ".".join(red_tokens[2:])

        return result

    # Reduced statistic:
    # pyspoc.rstatistics.pca.PCAEigenVectors.pca_eigenvectors_top_1_2_3.self_1
    if "pyspoc.rstatistics." in f:
        part = f.replace("pyspoc.rstatistics.", "")
        tokens = part.split(".")

        result["pipeline_type"] = "reduced_statistic"

        if len(tokens) >= 2:
            result["statistic_module"] = tokens[0]
            result["statistic_family"] = tokens[1]

            rest = tokens[2:]
            self_tokens = [x for x in rest if x.startswith("self")]
            non_self_tokens = [x for x in rest if not x.startswith("self")]

            result["statistic_output"] = ".".join(non_self_tokens)
            result["reducer_family"] = "self"
            result["reducer_output"] = ".".join(self_tokens)

        return result

    # Fallback: use first meaningful token
    tokens = re.split(r"[.:/\\]+", f)
    tokens = [t for t in tokens if t]

    result["pipeline_type"] = "unparsed"
    result["statistic_family"] = tokens[0] if tokens else "Other"
    result["reducer_family"] = "unknown"

    return result



Parsed 1214 features.
Saved feature-family mapping to: /Users/ilg/Desktop/year4/M4R/python_files/some_other_eda/loading_family_analysis/feature_family_mapping.csv

Pipeline type counts:
pipeline_type
statistic_reducer    838
reduced_statistic    303
unparsed              55
algorithm_metric      18
Name: count, dtype: int64

Top statistic families:
statistic_family
PCAEigenVectors              300
Covariance                   211
Precision                    211
KendallTau                   208
SpearmanR                    208
Users                         35
pyspoc                        20
Plaid metrics                  9
FABIA metrics                  9
PCAVarianceExplainedRatio      3
Name: count, dtype: int64

Top reducer families:
reducer_family
Moment                 800
self                   321
unknown                 55
EigenValues              8
SingularValues           8
Determinant              4
Diag                     4
EntryWiseMatrixNorm      4
Norm                  

In [ ]:
def main():
    df = pd.read_csv(FEATURES_PATH, nrows=1)
    features = list(df.columns)

    # Remove accidental index columns
    features = [
        f for f in features
        if not str(f).lower().startswith("unnamed")
        and str(f).lower() not in {"index", "level_0"}
    ]

    mapping = pd.DataFrame([parse_pyspoc_feature(f) for f in features])

    mapping.to_csv(OUT_PATH, index=False)

    print(f"Parsed {len(mapping)} features.")
    print(f"Saved feature-family mapping to: {OUT_PATH}")
    print("\nPipeline type counts:")
    print(mapping["pipeline_type"].value_counts())
    print("\nTop statistic families:")
    print(mapping["statistic_family"].value_counts().head(20))
    print("\nTop reducer families:")
    print(mapping["reducer_family"].value_counts().head(20))


main()

# over-representation ratio

In [ ]:
# ============================================================
# Paths
# ============================================================

PROJECT_DIR = Path("/Users/ilg/Desktop/year4/M4R/python_files")

MAPPING_PATH = PROJECT_DIR / "some_other_eda" / "loading_family_analysis" / "feature_family_mapping.csv"

LOADINGS_PATH = PROJECT_DIR / "eda_output_unnormalised" / "04_pca" / "pca_loadings.csv"

OUTPUT_DIR = PROJECT_DIR / "some_other_eda" / "loading_family_analysis" / "pc1_to_pc5_loading_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Settings
# ============================================================

TOP_PCS = [f"PC{i}" for i in range(1, 6)]
TOP_N_FEATURES_EACH_SIDE = 25
TOP_N_FAMILIES_FOR_PLOTS = 15


In [ ]:
# ============================================================
# Loading helpers
# ============================================================

def normalise_pc_name(x):
    s = str(x).strip().upper().replace(" ", "")
    if re.fullmatch(r"\d+", s):
        return f"PC{s}"
    return s


def load_feature_mapping(path: Path) -> pd.DataFrame:
    mapping = pd.read_csv(path)

    if "feature" not in mapping.columns:
        mapping = mapping.rename(columns={mapping.columns[0]: "feature"})

    required_cols = [
        "feature",
        "pipeline_type",
        "statistic_module",
        "statistic_family",
        "statistic_output",
        "reducer_module",
        "reducer_family",
        "reducer_output",
    ]

    for col in required_cols:
        if col not in mapping.columns:
            mapping[col] = "unknown"

    mapping["feature"] = mapping["feature"].astype(str)

    for col in required_cols[1:]:
        mapping[col] = mapping[col].fillna("unknown").astype(str)

    # A useful combined label for detailed inspection.
    mapping["statistic_reducer_family"] = (
        mapping["statistic_family"].astype(str)
        + " + "
        + mapping["reducer_family"].astype(str)
    )

    return mapping[required_cols + ["statistic_reducer_family"]]


def load_pca_loadings(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    # If feature is stored as an unnamed index column.
    if "feature" not in df.columns:
        possible_feature_cols = [
            c for c in df.columns
            if str(c).lower().startswith("unnamed")
            or str(c).lower() in {"index", "features"}
        ]

        if possible_feature_cols:
            df = df.rename(columns={possible_feature_cols[0]: "feature"})
        else:
            df = df.rename(columns={df.columns[0]: "feature"})

    df["feature"] = df["feature"].astype(str)

    # Normalise PC column names.
    rename_dict = {}
    for col in df.columns:
        if col == "feature":
            continue
        pc_name = normalise_pc_name(col)
        if pc_name.startswith("PC"):
            rename_dict[col] = pc_name

    df = df.rename(columns=rename_dict)

    missing_pcs = [pc for pc in TOP_PCS if pc not in df.columns]
    if missing_pcs:
        raise ValueError(
            f"Missing PC loading columns: {missing_pcs}. "
            f"Available columns are: {list(df.columns)[:20]} ..."
        )

    keep_cols = ["feature"] + TOP_PCS
    df = df[keep_cols].copy()

    for pc in TOP_PCS:
        df[pc] = pd.to_numeric(df[pc], errors="coerce")

    df = df.dropna(subset=TOP_PCS, how="all")

    return df


In [ ]:
# ============================================================
# Convert to long format
# ============================================================

mapping = load_feature_mapping(MAPPING_PATH)
loadings_wide = load_pca_loadings(LOADINGS_PATH)

loadings_long = loadings_wide.melt(
    id_vars="feature",
    value_vars=TOP_PCS,
    var_name="PC",
    value_name="loading",
)

loadings_long = loadings_long.dropna(subset=["loading"])

loadings_long = loadings_long.merge(
    mapping,
    on="feature",
    how="left",
)

# Fill missing mapping values.
mapping_cols = [
    "pipeline_type",
    "statistic_module",
    "statistic_family",
    "statistic_output",
    "reducer_module",
    "reducer_family",
    "reducer_output",
    "statistic_reducer_family",
]

for col in mapping_cols:
    loadings_long[col] = loadings_long[col].fillna("unmapped")

loadings_long["squared_loading"] = loadings_long["loading"] ** 2
loadings_long["direction"] = np.where(loadings_long["loading"] >= 0, "positive", "negative")

loadings_long.to_csv(OUTPUT_DIR / "loadings_long_pc1_to_pc5_with_mapping.csv", index=False)


# ============================================================
# Top signed individual features
# ============================================================

top_feature_rows = []

for pc in TOP_PCS:
    pc_df = loadings_long[loadings_long["PC"] == pc].copy()

    pos = (
        pc_df[pc_df["loading"] > 0]
        .sort_values("loading", ascending=False)
        .head(TOP_N_FEATURES_EACH_SIDE)
        .copy()
    )
    pos["rank_within_direction"] = range(1, len(pos) + 1)

    neg = (
        pc_df[pc_df["loading"] < 0]
        .sort_values("loading", ascending=True)
        .head(TOP_N_FEATURES_EACH_SIDE)
        .copy()
    )
    neg["rank_within_direction"] = range(1, len(neg) + 1)

    top_feature_rows.append(pos)
    top_feature_rows.append(neg)

top_features = pd.concat(top_feature_rows, ignore_index=True)
top_features.to_csv(OUTPUT_DIR / "top_25_positive_negative_loadings_pc1_to_pc5.csv", index=False)


In [ ]:
# ============================================================
# Family enrichment functions
# ============================================================

def compute_family_enrichment(df: pd.DataFrame, family_col: str) -> pd.DataFrame:
    """
    Computes:
        O_ik = family squared loading mass / PC total squared loading mass
        E_i  = family size / total number of features
        R_ik = O_ik / E_i
    using all loadings, not only top features.
    """
    unique_feature_family = df[["feature", family_col]].drop_duplicates()

    family_sizes = (
        unique_feature_family
        .groupby(family_col)
        .size()
        .rename("family_size")
        .reset_index()
    )

    p_total = unique_feature_family["feature"].nunique()

    pc_total_mass = (
        df.groupby("PC")["squared_loading"]
        .sum()
        .rename("pc_total_squared_loading")
        .reset_index()
    )

    family_pc_mass = (
        df.groupby(["PC", family_col])["squared_loading"]
        .sum()
        .rename("family_squared_loading")
        .reset_index()
    )

    out = (
        family_pc_mass
        .merge(pc_total_mass, on="PC", how="left")
        .merge(family_sizes, on=family_col, how="left")
    )

    out["p_total"] = p_total
    out["observed_share"] = out["family_squared_loading"] / out["pc_total_squared_loading"]
    out["expected_share"] = out["family_size"] / out["p_total"]
    out["enrichment_R"] = out["observed_share"] / out["expected_share"]
    out["log2_enrichment_R"] = np.log2(out["enrichment_R"])

    out = out.sort_values(["PC", "enrichment_R"], ascending=[True, False])

    return out


def compute_signed_family_enrichment(df: pd.DataFrame, family_col: str) -> pd.DataFrame:
    """
    Same enrichment idea, but separately within positive and negative loading directions.
    This is descriptive only because PC sign is arbitrary.
    """
    unique_feature_family = df[["feature", family_col]].drop_duplicates()

    family_sizes = (
        unique_feature_family
        .groupby(family_col)
        .size()
        .rename("family_size")
        .reset_index()
    )

    p_total = unique_feature_family["feature"].nunique()

    sign_total_mass = (
        df.groupby(["PC", "direction"])["squared_loading"]
        .sum()
        .rename("sign_total_squared_loading")
        .reset_index()
    )

    family_sign_mass = (
        df.groupby(["PC", "direction", family_col])["squared_loading"]
        .sum()
        .rename("family_sign_squared_loading")
        .reset_index()
    )

    out = (
        family_sign_mass
        .merge(sign_total_mass, on=["PC", "direction"], how="left")
        .merge(family_sizes, on=family_col, how="left")
    )

    out["p_total"] = p_total
    out["observed_share_within_sign"] = (
        out["family_sign_squared_loading"] / out["sign_total_squared_loading"]
    )
    out["expected_share"] = out["family_size"] / out["p_total"]
    out["signed_enrichment_R"] = out["observed_share_within_sign"] / out["expected_share"]
    out["log2_signed_enrichment_R"] = np.log2(out["signed_enrichment_R"])

    out = out.sort_values(
        ["PC", "direction", "signed_enrichment_R"],
        ascending=[True, True, False],
    )

    return out


def compute_direction_balance(df: pd.DataFrame, family_col: str) -> pd.DataFrame:
    """
    Measures whether a family contributes mainly to the positive or negative side:
        balance = (positive_mass - negative_mass) / total_mass.
    Values near 1 mean mostly positive; values near -1 mean mostly negative.
    """
    mass = (
        df.groupby(["PC", family_col, "direction"])["squared_loading"]
        .sum()
        .reset_index()
    )

    pivot = mass.pivot_table(
        index=["PC", family_col],
        columns="direction",
        values="squared_loading",
        fill_value=0.0,
    ).reset_index()

    if "positive" not in pivot.columns:
        pivot["positive"] = 0.0
    if "negative" not in pivot.columns:
        pivot["negative"] = 0.0

    pivot["total_squared_loading"] = pivot["positive"] + pivot["negative"]
    pivot["direction_balance"] = (
        (pivot["positive"] - pivot["negative"])
        / pivot["total_squared_loading"].replace(0, np.nan)
    )

    pivot = pivot.sort_values(["PC", "total_squared_loading"], ascending=[True, False])

    return pivot


In [ ]:
# ============================================================
# Run enrichment analyses
# ============================================================

family_columns = [
    "statistic_family",
    "reducer_family",
    "pipeline_type",
    "statistic_reducer_family",
]

for family_col in family_columns:
    enrichment = compute_family_enrichment(loadings_long, family_col)
    enrichment.to_csv(OUTPUT_DIR / f"enrichment_by_{family_col}.csv", index=False)

    signed_enrichment = compute_signed_family_enrichment(loadings_long, family_col)
    signed_enrichment.to_csv(OUTPUT_DIR / f"signed_enrichment_by_{family_col}.csv", index=False)

    direction_balance = compute_direction_balance(loadings_long, family_col)
    direction_balance.to_csv(OUTPUT_DIR / f"direction_balance_by_{family_col}.csv", index=False)

In [ ]:
# ============================================================
# Compact PC interpretation tables
# ============================================================

summary_rows = []

main_enrichment = compute_family_enrichment(loadings_long, "statistic_family")
main_signed = compute_signed_family_enrichment(loadings_long, "statistic_family")

for pc in TOP_PCS:
    pc_top_features = top_features[top_features["PC"] == pc].copy()

    top_pos = (
        pc_top_features[pc_top_features["direction"] == "positive"]
        .sort_values("loading", ascending=False)
        .head(5)
    )

    top_neg = (
        pc_top_features[pc_top_features["direction"] == "negative"]
        .sort_values("loading", ascending=True)
        .head(5)
    )

    top_families = (
        main_enrichment[main_enrichment["PC"] == pc]
        .sort_values("enrichment_R", ascending=False)
        .head(5)
    )

    for _, row in top_families.iterrows():
        summary_rows.append({
            "PC": pc,
            "summary_type": "top_enriched_statistic_family",
            "name": row["statistic_family"],
            "value": row["enrichment_R"],
            "observed_share": row["observed_share"],
            "expected_share": row["expected_share"],
            "family_size": row["family_size"],
        })

    for _, row in top_pos.iterrows():
        summary_rows.append({
            "PC": pc,
            "summary_type": "top_positive_feature",
            "name": row["feature"],
            "value": row["loading"],
            "statistic_family": row["statistic_family"],
            "reducer_family": row["reducer_family"],
        })

    for _, row in top_neg.iterrows():
        summary_rows.append({
            "PC": pc,
            "summary_type": "top_negative_feature",
            "name": row["feature"],
            "value": row["loading"],
            "statistic_family": row["statistic_family"],
            "reducer_family": row["reducer_family"],
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "pc1_to_pc5_interpretation_summary.csv", index=False)

In [ ]:
# ============================================================
# Plots
# ============================================================

def plot_enrichment(enrichment_df: pd.DataFrame, family_col: str, pc: str):
    sub = enrichment_df[enrichment_df["PC"] == pc].copy()
    sub = sub.sort_values("enrichment_R", ascending=False).head(TOP_N_FAMILIES_FOR_PLOTS)

    if sub.empty:
        return

    plt.figure(figsize=(9, 5))
    plt.barh(sub[family_col].astype(str), sub["enrichment_R"])
    plt.axvline(1.0, linestyle="--", linewidth=1)
    plt.gca().invert_yaxis()
    plt.xlabel("Enrichment ratio")
    plt.ylabel(family_col)
    plt.title(f"{pc}: enrichment by {family_col}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{pc.lower()}_enrichment_by_{family_col}.png", dpi=200)
    plt.close()


def plot_top_signed_features(pc: str):
    sub = top_features[top_features["PC"] == pc].copy()

    pos = (
        sub[sub["direction"] == "positive"]
        .sort_values("loading", ascending=False)
        .head(10)
    )

    neg = (
        sub[sub["direction"] == "negative"]
        .sort_values("loading", ascending=True)
        .head(10)
    )

    plot_df = pd.concat([neg, pos], ignore_index=True)

    if plot_df.empty:
        return

    labels = []
    for _, row in plot_df.iterrows():
        label = f"{row['statistic_family']} + {row['reducer_family']}"
        labels.append(label[:80])

    plt.figure(figsize=(10, 7))
    plt.barh(labels, plot_df["loading"])
    plt.axvline(0.0, linewidth=1)
    plt.xlabel("Loading")
    plt.ylabel("Feature family + reducer")
    plt.title(f"{pc}: top signed feature loadings")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{pc.lower()}_top_signed_feature_loadings.png", dpi=200)
    plt.close()


for pc in TOP_PCS:
    stat_enrichment = compute_family_enrichment(loadings_long, "statistic_family")
    red_enrichment = compute_family_enrichment(loadings_long, "reducer_family")

    plot_enrichment(stat_enrichment, "statistic_family", pc)
    plot_enrichment(red_enrichment, "reducer_family", pc)
    plot_top_signed_features(pc)


# ============================================================
# Markdown report
# ============================================================

unmatched_count = (loadings_long["pipeline_type"] == "unmapped").sum()
n_features = loadings_long["feature"].nunique()

report_lines = []

report_lines.append("# PC1--PC5 loading-family analysis\n\n")
report_lines.append(f"- Number of unique features: `{n_features}`\n")
report_lines.append(f"- Number of unmapped loading entries: `{unmatched_count}`\n\n")

report_lines.append("## Interpretation note\n\n")
report_lines.append(
    "The main family analysis uses `statistic_family`, which records the statistic or reduced statistic "
    "that generated the feature. For statistic--reducer pipelines, the reducer is analysed separately as "
    "`reducer_family`. This avoids forcing a feature such as `Covariance + Moment` into only one conceptual "
    "category.\n\n"
)
report_lines.append(
    "The enrichment score is descriptive. It adjusts for family size by comparing observed squared-loading "
    "share with the expected share under equal per-feature contribution. Positive and negative loading signs "
    "are interpreted as opposite sides of the same PC axis, since the absolute sign of a principal component "
    "is arbitrary.\n\n"
)

stat_enrichment = compute_family_enrichment(loadings_long, "statistic_family")

for pc in TOP_PCS:
    report_lines.append(f"## {pc}\n\n")

    report_lines.append("### Top enriched statistic families\n\n")
    sub_fam = (
        stat_enrichment[stat_enrichment["PC"] == pc]
        .sort_values("enrichment_R", ascending=False)
        .head(8)
    )

    for _, row in sub_fam.iterrows():
        report_lines.append(
            f"- `{row['statistic_family']}`: "
            f"R = `{row['enrichment_R']:.3f}`, "
            f"observed share = `{row['observed_share']:.3f}`, "
            f"expected share = `{row['expected_share']:.3f}`, "
            f"family size = `{int(row['family_size'])}`\n"
        )

    report_lines.append("\n### Top positive individual features\n\n")
    pos = (
        top_features[
            (top_features["PC"] == pc)
            & (top_features["direction"] == "positive")
        ]
        .sort_values("loading", ascending=False)
        .head(8)
    )

    for _, row in pos.iterrows():
        report_lines.append(
            f"- `{row['statistic_family']} + {row['reducer_family']}`: "
            f"loading = `{row['loading']:.4f}`; "
            f"`{row['feature']}`\n"
        )

    report_lines.append("\n### Top negative individual features\n\n")
    neg = (
        top_features[
            (top_features["PC"] == pc)
            & (top_features["direction"] == "negative")
        ]
        .sort_values("loading", ascending=True)
        .head(8)
    )

    for _, row in neg.iterrows():
        report_lines.append(
            f"- `{row['statistic_family']} + {row['reducer_family']}`: "
            f"loading = `{row['loading']:.4f}`; "
            f"`{row['feature']}`\n"
        )

    report_lines.append("\n")

REPORT_PATH = OUTPUT_DIR / "pc1_to_pc5_loading_family_report.md"
REPORT_PATH.write_text("".join(report_lines))


print("Done.")
print(f"Outputs saved to: {OUTPUT_DIR}")
print(f"Main long loadings file: {OUTPUT_DIR / 'loadings_long_pc1_to_pc5_with_mapping.csv'}")
print(f"Top signed loadings: {OUTPUT_DIR / 'top_25_positive_negative_loadings_pc1_to_pc5.csv'}")
print(f"Statistic-family enrichment: {OUTPUT_DIR / 'enrichment_by_statistic_family.csv'}")
print(f"Reducer-family enrichment: {OUTPUT_DIR / 'enrichment_by_reducer_family.csv'}")
print(f"Interpretation summary: {OUTPUT_DIR / 'pc1_to_pc5_interpretation_summary.csv'}")
print(f"Markdown report: {REPORT_PATH}")

Done.
Outputs saved to: /Users/ilg/Desktop/year4/M4R/python_files/some_other_eda/loading_family_analysis/pc1_to_pc5_loading_analysis
Main long loadings file: /Users/ilg/Desktop/year4/M4R/python_files/some_other_eda/loading_family_analysis/pc1_to_pc5_loading_analysis/loadings_long_pc1_to_pc5_with_mapping.csv
Top signed loadings: /Users/ilg/Desktop/year4/M4R/python_files/some_other_eda/loading_family_analysis/pc1_to_pc5_loading_analysis/top_25_positive_negative_loadings_pc1_to_pc5.csv
Statistic-family enrichment: /Users/ilg/Desktop/year4/M4R/python_files/some_other_eda/loading_family_analysis/pc1_to_pc5_loading_analysis/enrichment_by_statistic_family.csv
Reducer-family enrichment: /Users/ilg/Desktop/year4/M4R/python_files/some_other_eda/loading_family_analysis/pc1_to_pc5_loading_analysis/enrichment_by_reducer_family.csv
Interpretation summary: /Users/ilg/Desktop/year4/M4R/python_files/some_other_eda/loading_family_analysis/pc1_to_pc5_loading_analysis/pc1_to_pc5_interpretation_summary.csv